# Snowflake DCM Projects (PrPr) - Quickstart 2 (Workspaces)

## Project Files

This Quickstart contains two demo DCM Projects for you to explore how both use-cases can be combined:
- **DCM_Platform_Demo** — Platform infrastructure (Warehouse, Roles, Grants, ingestion with a Task loading csv files from a Stage into raw tables using a defined File Format)
- **DCM_Pipeline_Demo** — Data transformation pipeline (Dynamic Tables, Views, Data Quality Expectations)

Explore the content of the manifest files and the various definition files:
- The Manifest contains DEV and PROD target profiles and corresponding templating configurations
- The definition files contain various DEFINE statements for new entities, as well as GRANTS, ATTACH DMF and examples for jinja templating
- The macros folder is optional and can store global jinja macros to be used across different definition files

**Tip:** Use the split-screen to keep manifest and definition files left and this setup file on the right

#

## Role Setup

**Step 1** (optional, but recommended): 

Create a dedicated dcm-admin role to manage DCM Projects on a DEV environment

In [ ]:
%%sql -r role_setup
use role ACCOUNTADMIN;

create role if not exists DCM_DEVELOPER;
set USER_NAME = (select current_user());
grant role DCM_DEVELOPER to user identifier($USER_NAME);

**Step 2:** Grant permissions to create new account infrastructure through DCM deployments
(depending on your needs and project purpose)

In [ ]:
%%sql -r grant_infrastructure
grant CREATE WAREHOUSE on account to role DCM_DEVELOPER;
grant CREATE ROLE on account to role DCM_DEVELOPER;
grant CREATE DATABASE on account to role DCM_DEVELOPER;
grant EXECUTE MANAGED TASK on account to role DCM_DEVELOPER;
grant EXECUTE TASK on account to role DCM_DEVELOPER;

grant MANAGE GRANTS on account to role DCM_DEVELOPER;

**Step 3:** If you want to define and test data quality expectations you also need:

In [ ]:
%%sql -r grant_dq
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_DEVELOPER;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_DEVELOPER;
grant database role SNOWFLAKE.DATA_METRIC_USER to role DCM_DEVELOPER;
grant execute data metric function on account to role DCM_DEVELOPER with grant option;

## Platform Project Setup

**Step 1** (optional): In case you don't have a warehouse available or want to run the deployments on a dedicated warehouse

(DCM commands are mostly meta-data changes, which makes them very lightweight. Only few commands actually require a warehouse and XS is sufficient)

In [ ]:
%%sql -r create_wh
use role DCM_DEVELOPER;

create warehouse if not exists DCM_WH
with
    warehouse_size = 'XSMALL'
    auto_suspend = 300
    comment = 'For Quickstart Demo of DCM Projects'
;

**Step 2:** Create the DCM Project Object inside a schema that executes and stores all deployments of your project definitions

In [ ]:
%%sql -r create_db_schema
create database if not exists DCM_DEMO;
create schema if not exists DCM_DEMO.PROJECTS;


create dcm project if not exists DCM_DEMO.PROJECTS.DCM_PLATFORM_DEV
    comment = 'for DCM Platform Demo - Quickstart 2'
;

## Dry-run a deployment using the PLAN command

**1. In the DCM control panel above the tabs, select the project `DCM_PLATFORM_DEV`**

- See how the `DCM_DEV` target is already pre-selected as it is set as default in the manifest
- Click on the target profile to see that it uses the previously created DCM_PLATFORM_DEV object and the `DEV` templating configuration
- Override the templating value for `users` with your own user-name. (users are defined as a list, so format as ['MY_USERNAME'])
- Optional: open the manifest file to see what other values this configuration contains

**2. Execute a "Plan" to the DCM_DEV target environment**
- Click ▶️ and wait for the definitions to render, compile and dry-run the statements

**3. Review the plan result** 
- Review the operations, object names and types that will be deployed
- expand sections to see more details

## Deploy the Project Definitions

If the plan result was successful and all planned changes are correct, you can now safely `Deploy` those changes.

- Change the DCM operation from PLAN to DEPLOY and click ▶️.
- Optional: You can add a Deployment-alias which you can think of as a "commit name" to see later in the deployment history.
- DCM will create all the objects and attach grants and expectations using the owner-role of the project object.
- Once the deployment completed successfully, you can refresh the Database Explorer on the left to see the created database and all objects inside.

## Load Sample Data

Now that the INGEST schema was created, we can copy some sample CSV files from the workspace to the ingestion stage and trigger the load task.

In [ ]:
%%sql -r stage_result
-- Copy all CSV files from demo repo to the stage

COPY FILES INTO 
    @DCM_DEMO_2_DEV.INGEST.DCM_SAMPLE_DATA
FROM 
    'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_2/sample_data'
DETAILED_OUTPUT = TRUE
;

Now that we have some new data in our source stage we can manually trigger the Task to load that new data into our RAW tables.

In [ ]:
%%sql -r load_data
execute task DCM_DEMO_2_DEV.INGEST.LOAD_NEW_DATA;

## Deploying the PROD Instance

Now that you have successfully deployed and tested the DEV instance of the platform project, you can deploy the PROD instance.

1. Review the manifest.yml to see how the templating values differ between DEV and PROD.
    - Due to the naming suffix both targets can co-exist on the same Snowflake account (but you could just as well deploy them on separate accounts).

In [ ]:
%%sql -r grant_privileges_to_prod_deployer
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_PROD_DEPLOYER;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_PROD_DEPLOYER;
grant database role SNOWFLAKE.DATA_METRIC_USER to role DCM_PROD_DEPLOYER;
grant execute data metric function on account to role DCM_PROD_DEPLOYER with grant option;

2. PLAN against PROD
    - In the Workspace's DCM action bar switch to the PROD target and run PLAN to preview what will be deployed as PROD instance.
    - Notice the warning that the `DCM_DEMO.PROJECTS.DCM_PLATFORM` object for the PROD target does not exist yet. 
    - If you run PLAN anyway Snowflake Workspaces will offer to create the project object for you - with the owner-role that is configured in the manifest.    
    - If the PLAN was successful and the definitions are what you expect, then DEPLOY the platform project to production.
3. Load sample data to PROD stage
    - As a last step you load the sample data into the PROD stage. Run the command below.

In [ ]:
%%sql -r dataframe_2
-- Copy all CSV files from demo repo to the stage

COPY FILES INTO 
    @DCM_DEMO_2.INGEST.DCM_SAMPLE_DATA
FROM 
    'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_2/sample_data'
DETAILED_OUTPUT = TRUE
;

In [ ]:
%%sql -r dataframe_1
-- manually trigger ingestion task on PROD

execute task DCM_DEMO_2.INGEST.LOAD_NEW_DATA;

---
# DCM Projects for Pipelines

You can leverage additional DCM capabilities when creating and deploying transformation pipelines that leverage Dynamic Tables and Data Metric Functions.

## Pipeline Project Setup

* The DCM_DEVELOPER has already created a DCM_DEMO_2_FINANCE_DEV_ADMIN role as part of deploying the DCM Platform Project.
* The DCM_DEMO_2_FINANCE_DEV_ADMIN has also been granted specific privileges that are needed to now operate on the DCM_DEMO_2_FINANCE_DEV database.
* Now the DCM_DEMO_2_FINANCE_DEV_ADMIN can create its own DCM Project inside this database to manage a new transformation pipeline
* The DCM_DEMO_2_FINANCE_DEV_ADMIN also has been granted SELECT privileges on the RAW tables. So it can now use these as a data source for the transformation pipeline.

In [ ]:
%%sql -r grant_finance_dq
-- because DCM can not yet grant APPLICATION ROLES we need to manually grant this here for now  

use role DCM_DEVELOPER;

grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_DEMO_2_FINANCE_DEV_ADMIN;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_DEMO_2_FINANCE_DEV_ADMIN;

In [ ]:
%%sql -r create_pipeline_project
use role DCM_DEMO_2_FINANCE_DEV_ADMIN;

create or replace dcm project DCM_DEMO_2_FINANCE_DEV.PROJECTS.FINANCE_PIPELINE
    comment = 'for DCM Pipeline Demos - Quickstart 2'
;

### PLAN and DEPLOY
Now in the shoes of the DCM_DEMO_2_FINANCE_DEV_ADMIN, you can test the definitions of the Pipeline project with the DEV configuration against the current state on the account, and then deploy.

1. In the DCM control panel above the tabs, select the project `FINANCE_PIPELINE`

2. Execute a "Plan" to the DCM_DEV target environment
    - Click ▶️ and wait for the definitions to render, compile and dry-run the statements

3. Review the plan result
    - if the plan was successful, you can see the objects that will be created by the DCM_DEMO_2_FINANCE_DEV_ADMIN inside the DCM_DEMO_2_FINANCE_DEV database

4. Execute the deployment of the Pipeline project*
    - Refresh the Database explorer on the left to see the new objects that were created 

### REFRESH Command

Now that the pipeline project was created you can refresh all Dynamic Tables defined in the pipeline project to process the new data from the RAW tables.

Trigger a manual refresh now so you don't have to wait for the scheduled refresh.

In [ ]:
%%sql -r refresh_result
execute dcm project DCM_DEMO_2_FINANCE_DEV.PROJECTS.FINANCE_PIPELINE 
    REFRESH ALL

    -- optional flow operator to format the json output as table
    ->> select
            f.value:data_timestamp ::string as TIMESTAMP_IN_MS,
            f.value:dt_name ::string as DT_NAME,
            f.value:statistics ::string as RESULT,
            f.value:refreshed_dt_count ::number as REFRESHED_DTS
        from
            $1 as REFRESH_RESULT,
            lateral flatten(input => parse_json(REFRESH_RESULT."result"):refreshed_tables) f
;

## TEST Command

Now that the sample data was loaded and processed throughout the transformation pipeline, you can test all Data Quality Expectations that were set on table-objects inside the project.

Run the command below to see if all expectations are met:

In [ ]:
%%sql -r pipeline_test_results
execute dcm project DCM_DEMO_2_FINANCE_DEV.PROJECTS.FINANCE_PIPELINE 
    TEST ALL

    -- optional flow operator to format the json output as table
    ->> select
            f.value:table_name ::string as TABLE_NAME,
            f.value:metric_database ::string as METRIC_DATABASE,
            f.value:metric_schema ::string as METRIC_SCHEMA,
            f.value:metric_name ::string as METRIC_NAME,
            f.value:expectation_name ::string as EXPECTATION_NAME,
            f.value:expectation_expression ::string as EXPECTATION_EXPRESSION,
            case 
                when f.value:expectation_violated ::boolean = 'FALSE' then '✅ MET'
                when f.value:expectation_violated ::boolean = 'TRUE' then '🚫 FAILED'
                end as EXPECTATION_RESULT,
            f.value:value ::number as VALUE
        from
            $1 as TEST_RESULTS,
            lateral flatten(input => parse_json(TEST_RESULTS."message"):expectations) f
;

## PREVIEW Command

If you are making changes to pipeline definitions then you can preview a data-sample even before deploying the change.

Preview can be run for a defined table / dynamic table / view.

For example you can change the definition of a Dynamic Table, then run the PREVIEW command to see how the change impacts the downstream data.

Note: PREVIEW is recommended for DEV environments that contain smaller datasets. It will be too slow to run on full-scale production dataset.

In [ ]:
%%sql -r preview_result
execute dcm project DCM_DEMO_2_FINANCE_DEV.PROJECTS.FINANCE_PIPELINE
    PREVIEW DCM_DEMO_2_FINANCE_DEV.GOLD.FACT_PROSPECT
    using configuration DEV
    from
        'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_2/DCM_Pipeline_Demo'
;

---
# Plan and Deploy the Pipeline to Production

By now you should have all the experience to know how you can deploy the Finance Pipeline to the Production environment. 

If you have more questions or issues to debug, try asking Cortex Code for help or review the product documentation on https://docs.snowflake.com/en/user-guide/dcm-projects/dcm-projects-overview 